![alt text](icon.png) 

# Pivotal demo

Demo of the Pivotal language and Jupyter Lab extension. Pivotal source code and docs avaialble on [GitHub](https://github.com/nealbob/pivotal-py). Data is from the Kaggle [European Soccer Database](https://www.kaggle.com/datasets/hugomathien/soccer). 

The outputs from this notebook are saved in ```/football_demo/``` (visible from the Jupyter Lab file explorer on the left). There is also an exported Python file ```football_demo.py``` containing the compiled Pandas code. 


In [1]:
import pivotal
%pivotal_set canvas=a4, viewer_font=0.9
#%pivotal_set backend=duckdb

Pivotal settings: {'viewer': True, 'output_code': False, 'canvas': 'a4', 'margins': 25.4, 'chart_width': 'full', 'viewer_font': 0.9, 'viewer_num_format': 5, 'use_visions': False, 'backend': 'pandas'}


In [2]:
big4 = ["England Premier League", "Spain LIGA BBVA", "Germany 1. Bundesliga", "Italy Serie A"]

In [3]:
%%pivotal 
load Match "database.sqlite"
load League "database.sqlite"

# Data can also be loaded via gui 
# pivotal.load_gui()

In [4]:
%%pivotal
df league_names from League
    filter name in :big4
    rename id as league_id, name as League
    select league_id, League

df Match
    inner merge league_names on league_id
    season_clean = left(season,4) + "-" + right(season,2)
    total_goals = home_team_goal + away_team_goal

In [5]:
%%pivotal
# Mean scoring level by league

df goal_summary from Match
    pivot
        agg mean total_goals
        rows season_clean
        cols League
    plot line goal_chart
        x season_clean "Season"
        y :big4 "Mean goals"

# Same chart but with agg plot (built in pivot)
df Match  
    agg plot line goal_chart2
        x season_clean "Season"
        y mean total_goals "Mean goals"
        by League           # single plot with legend if only 1 y var

# plots can also be generated via gui
# pivotal.plot_gui()

In [6]:
%%pivotal
df goal_summary  # Optional as this df allready has focus
    table goal_table
        title "Mean goals scored per game by league and season"
        format number 2
        stub season_clean "Season"
        summary mean as "Average"
        font size 11
        font "Calibri"
        show

In [7]:
%%pivotal
load Team "database.sqlite"
    select team_api_id, team_long_name

In [8]:
%%pivotal
# Teams with most wins

df win_summary from Match
    winner_id = home_team_api_id 
        where home_team_goal > away_team_goal
    winner_id = away_team_api_id
        where home_team_goal < away_team_goal
    left merge Team
         left_on winner_id
         right_on team_api_id
    wins = 1
    group by team_long_name, League
        agg sum wins
    select team_long_name as Team, League, wins as Wins 
    sort Wins asc
    filter Wins > 50

In [9]:
%%pivotal
df win_summary
    plot barh win_plot
        x Team 
        y Wins ""
        by League
        legend False

In [10]:
%%pivotal
# Combine home and away results 
df homematch from Match
    GD = home_team_goal - away_team_goal
    select home_team_api_id as team_id, date, GD
    sort team_id, date

df ha_match from Match
    GD = away_team_goal - home_team_goal
    select away_team_api_id as team_id, date, GD
    sort team_id, date
    concat homematch
    rolling mean GD 4 as pastgame_GD 
        order date
        by team_id 

delete homematch

In [11]:
# Switch to python if convenient

ha_match['form'] = pd.qcut(ha_match['pastgame_GD'], q=5)

# Update object viewer and explorer panels
pivotal.update()

[Pivotal] Updated viewer: Match, League, league_names, goal_summary, goal_chart2_df, Team, win_summary, ha_match


In [12]:
%%pivotal
# Link between past team form and match outcomes
df ha_match 
    group by form
        agg mean GD
        
    plot bar form_chart
        title "Prior form vs match outcome"
        x form "Past four games mean goal difference"
        y GD "Mean goal difference"
        legend False

In [13]:
%%pivotal
save "football_demo"

Package 'football_demo' saved to C:\Code_win\pivotal-demo\football_demo (12 dataframe(s), 4 chart(s), 1 table(s))
